In [0]:
# Databricks notebook: 05_map_users
# Purpose: Transform PhonePe map user JSON into an
# analytics-ready state/district-level Silver Delta table.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    MapType,
    StringType,
    BooleanType,
    LongType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
SILVER_TABLE = "workspace.silver.map_users"

# 34 country files + 1,224 state files
EXPECTED_SOURCE_FILES = 1_258

print("Configuration")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")


# ============================================================
# 2. SELECT MAP USER SOURCE FILES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE).alias("b")

map_user_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "map") &
        (F.col("domain") == "user")
    )
    .alias("m")
)

map_user_source_df = (
    map_user_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

actual_source_files = (
    map_user_source_df
    .select("source_path")
    .distinct()
    .count()
)

print("\nSource validation")
print(f"Expected source files: {EXPECTED_SOURCE_FILES:,}")
print(f"Actual source files  : {actual_source_files:,}")

assert actual_source_files == EXPECTED_SOURCE_FILES, (
    f"Expected {EXPECTED_SOURCE_FILES:,} files, "
    f"but found {actual_source_files:,}."
)

print("Map user source validation passed.")


# ============================================================
# 3. DEFINE SCHEMA
# Supports both older and newer PhonePe field names.
# ============================================================

area_user_metrics_schema = StructType([
    StructField("registeredUsers", LongType(), True),
    StructField("registeredCount", LongType(), True),
    StructField("appOpens", LongType(), True)
])

map_user_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),

    StructField(
        "data",
        StructType([
            StructField(
                "hoverData",
                MapType(
                    StringType(),
                    area_user_metrics_schema,
                    True
                ),
                True
            )
        ]),
        True
    ),

    StructField("responseTimestamp", LongType(), True)
])

print("\nExplicit map user schema created.")


# ============================================================
# 4. PARSE JSON
# ============================================================

parsed_map_user_df = (
    map_user_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            map_user_schema
        )
    )
)

parse_summary = (
    parsed_map_user_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col("parsed_json.data.hoverData").isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_map_user_data")
    )
    .first()
)

print("\nJSON parsing")
print(f"Source files               : {parse_summary['source_files']:,}")
print(
    f"Files without map user data: "
    f"{parse_summary['files_without_map_user_data']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_map_user_data"] == 0, (
    "One or more files could not be parsed or do not "
    "contain hoverData."
)

print("Map user JSON parsing passed.")


# ============================================================
# 5. FLATTEN DYNAMIC GEOGRAPHY MAP
# ============================================================

map_users_with_optional_fields_df = (
    parsed_map_user_df

    # Convert the hoverData map into key/value records.
    .withColumn(
        "area_entry",
        F.explode(
            F.map_entries(
                F.col("parsed_json.data.hoverData")
            )
        )
    )

    .select(
        F.col("file_scope"),

        F.when(
            F.col("file_scope") == "country",
            F.lit("state")
        )
        .otherwise(F.lit("district"))
        .alias("geography_level"),

        F.when(
            F.col("file_scope") == "state",
            F.col("state_slug")
        )
        .otherwise(F.lit(None).cast("string"))
        .alias("parent_state_slug"),

        F.trim(
            F.col("area_entry.key")
        ).alias("area_name"),

        F.lower(
            F.regexp_replace(
                F.trim(F.col("area_entry.key")),
                "[^a-zA-Z0-9]+",
                "-"
            )
        ).alias("area_slug"),

        F.col("year").cast("int").alias("year"),
        F.col("quarter").cast("int").alias("quarter"),
        F.col("period"),

        # Handles either possible source-field name.
        F.coalesce(
            F.col("area_entry.value.registeredUsers"),
            F.col("area_entry.value.registeredCount")
        )
        .cast("long")
        .alias("registered_user_count"),

        F.when(
            F.col("area_entry.value.registeredUsers").isNotNull(),
            F.lit("registeredUsers")
        )
        .when(
            F.col("area_entry.value.registeredCount").isNotNull(),
            F.lit("registeredCount")
        )
        .otherwise(F.lit(None))
        .alias("registered_count_source_field"),

        F.col("area_entry.value.appOpens")
            .cast("long")
            .alias("app_opens"),

        F.col("parsed_json.responseTimestamp")
            .alias("response_timestamp_epoch_ms"),

        F.col("relative_path"),
        F.col("source_path"),
        F.col("source_commit"),
        F.col("content_sha256"),
        F.col("ingested_at_utc")
    )
)

print("\nDynamic geographic user records flattened.")


# ============================================================
# 6. HANDLE OPTIONAL APP-OPEN FIELD
# ============================================================

populated_app_open_rows = (
    map_users_with_optional_fields_df
    .filter(F.col("app_opens").isNotNull())
    .count()
)

if populated_app_open_rows == 0:
    map_users_final_df = (
        map_users_with_optional_fields_df
        .drop("app_opens")
    )

    print(
        "app_opens removed because it is absent from "
        "every map user record."
    )
else:
    map_users_final_df = map_users_with_optional_fields_df

    print(
        f"app_opens retained: "
        f"{populated_app_open_rows:,} populated records."
    )


# ============================================================
# 7. DATA-QUALITY VALIDATION
# ============================================================

quality_result = (
    map_users_final_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("area_name").isNull() |
                F.col("geography_level").isNull() |
                F.col("registered_user_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_required_values"),

        F.sum(
            F.when(
                F.col("registered_user_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_registered_user_counts"),

        F.countDistinct("source_path")
            .alias("represented_source_files"),

        F.min("registered_user_count")
            .alias("minimum_registered_users"),

        F.max("registered_user_count")
            .alias("maximum_registered_users")
    )
    .first()
)

duplicate_business_keys = (
    map_users_final_df
    .groupBy(
        "source_path",
        "area_name"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nData-quality results")
print(f"Total Silver rows             : {quality_result['total_rows']:,}")
print(
    f"Missing required values       : "
    f"{quality_result['missing_required_values']:,}"
)
print(
    f"Negative registered-user count: "
    f"{quality_result['negative_registered_user_counts']:,}"
)
print(
    f"Source files represented      : "
    f"{quality_result['represented_source_files']:,}"
)
print(
    f"Minimum registered users      : "
    f"{quality_result['minimum_registered_users']:,}"
)
print(
    f"Maximum registered users      : "
    f"{quality_result['maximum_registered_users']:,}"
)
print(f"Duplicate business keys       : {duplicate_business_keys:,}")

assert quality_result["total_rows"] > 0
assert quality_result["missing_required_values"] == 0
assert quality_result["negative_registered_user_counts"] == 0
assert quality_result["represented_source_files"] == EXPECTED_SOURCE_FILES
assert duplicate_business_keys == 0

print("Map user quality checks passed.")


# ============================================================
# 8. DISPLAY SOURCE FIELD USAGE
# ============================================================

print("\nRegistered-user source-field usage")

display(
    map_users_final_df
    .groupBy("registered_count_source_field")
    .agg(
        F.count("*").alias("row_count")
    )
    .orderBy(
        F.desc("row_count")
    )
)


# ============================================================
# 9. SAVE MANAGED DELTA TABLE
# ============================================================

(
    map_users_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\nSilver Delta table created: {SILVER_TABLE}")


# ============================================================
# 10. VALIDATE SAVED TABLE
# ============================================================

saved_map_users_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_map_users_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period")
    )
    .first()
)

print("\nSaved-table validation")
print(f"Saved rows              : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period            : {saved_summary['first_period']}")
print(f"Last period             : {saved_summary['last_period']}")

assert saved_summary["row_count"] > 0
assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final map user Delta-table validation passed.")


# ============================================================
# 11. DISPLAY SAMPLE
# ============================================================

display(
    saved_map_users_df
    .select(
        "geography_level",
        "parent_state_slug",
        "area_name",
        "year",
        "quarter",
        "period",
        "registered_user_count",
        "registered_count_source_field"
    )
    .orderBy(
        "year",
        "quarter",
        "geography_level",
        "parent_state_slug",
        "area_name"
    )
    .limit(20)
)

Configuration
Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.map_users
Expected files: 1,258

Source validation
Expected source files: 1,258
Actual source files  : 1,258
Map user source validation passed.

Explicit map user schema created.

JSON parsing
Source files               : 1,258
Files without map user data: 0
Map user JSON parsing passed.

Dynamic geographic user records flattened.
app_opens removed because it is absent from every map user record.

Data-quality results
Total Silver rows             : 27,846
Missing required values       : 0
Negative registered-user count: 0
Source files represented      : 1,258
Minimum registered users      : 1
Maximum registered users      : 90,563,992
Duplicate business keys       : 0
Map user quality checks passed.

Registered-user source-field usage


registered_count_source_field,row_count
registeredCount,27846



Silver Delta table created: workspace.silver.map_users

Saved-table validation
Saved rows              : 27,846
Source files represented: 1,258
First period            : 2018-Q1
Last period             : 2026-Q2
Final map user Delta-table validation passed.


geography_level,parent_state_slug,area_name,year,quarter,period,registered_user_count,registered_count_source_field
district,andaman-&-nicobar-islands,nicobar district,2018,1,2018-Q1,484,registeredCount
district,andaman-&-nicobar-islands,north and middle andaman district,2018,1,2018-Q1,879,registeredCount
district,andaman-&-nicobar-islands,south andaman district,2018,1,2018-Q1,7929,registeredCount
district,andhra-pradesh,alluri sitharama raju district,2018,1,2018-Q1,15140,registeredCount
district,andhra-pradesh,anakapalli district,2018,1,2018-Q1,99229,registeredCount
district,andhra-pradesh,ananthapuramu district,2018,1,2018-Q1,137680,registeredCount
district,andhra-pradesh,annamayya district,2018,1,2018-Q1,80883,registeredCount
district,andhra-pradesh,bapatla district,2018,1,2018-Q1,64000,registeredCount
district,andhra-pradesh,chittoor district,2018,1,2018-Q1,84412,registeredCount
district,andhra-pradesh,dr br ambedkar konaseema district,2018,1,2018-Q1,72985,registeredCount


# Silver Map Users — Completion Summary

The `workspace.silver.map_users` Delta table was successfully created and validated.

## Table grain

Each record represents one geographical area and quarter:

- Country-level source files produce state-level records.
- State-level source files produce district-level records.

## Available metrics

- Registered-user count
- State or district geography
- Year and quarter
- Original source-field name
- Complete source lineage

## Source-schema handling

The pipeline supports both `registeredUsers` and `registeredCount`. The current pinned source version uses `registeredCount`.

The source does not provide app-open information, so the completely empty `app_opens` column was excluded.

## Analytical interpretation

Registered-user count is cumulative. It must not be summed across quarters.

The Gold/dbt layer will calculate:

- Quarterly user additions
- Quarter-over-quarter growth
- Transactions per registered user
- Transaction value per registered user
- Regional digital-adoption indicators